# 04 — Model Training
### Chicago Housing Market | Listing Risk Prediction

Now that our data is clean and our features are engineered, it's time to **train machine learning models**.

We'll train four models:
- **Regression:** Linear Regression + Random Forest Regressor → predict `RISK_SCORE` (a continuous number)
- **Classification:** Logistic Regression + Random Forest Classifier → predict `RISK_LABEL` (high/low risk binary)

**Input:** `data/processed/chicago_housing_engineered.csv`  
**Output:** 4 trained models + 1 scaler saved to `models/`

### What is Supervised Machine Learning?

**Supervised learning** means we give the model example inputs and their correct answers (labels), and it learns the mapping between them.

There are two types of supervised tasks:

| Task | Output | Example |
|------|--------|--------|
| **Regression** | A number | Predict tomorrow's temperature (23.5°C) |
| **Classification** | A category | Predict if it will rain (Yes/No) |

For listing risk prediction:
- **Regression** → predict the exact `RISK_SCORE` value (e.g. 1.32)
- **Classification** → predict whether `RISK_LABEL` is 0 (low risk) or 1 (high risk)

We train **two models per task** to compare a simple model (linear) against a more complex one (Random Forest).

### What are we doing in this cell?

We import scikit-learn — the most popular Python machine learning library. Each model is a class we import and then call `.fit()` on to train it.

We also import `joblib` which lets us save trained models to disk so we don't have to re-train them every time.

In [29]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
import joblib
import os

df = pd.read_csv('../data/processed/chicago_housing_engineered.csv')
df['PERIOD_BEGIN'] = pd.to_datetime(df['PERIOD_BEGIN'])

print(f'Loaded: {df.shape}')
df.head(3)

Loaded: (155, 41)


,PERIOD_BEGIN,MEDIAN_SALE_PRICE,HOMES_SOLD,NEW_LISTINGS,INVENTORY,MONTHS_OF_SUPPLY,MEDIAN_DOM,PRICE_DROPS,MONTH,YEAR,...,INVENTORY_ROLL6,MONTHS_OF_SUPPLY_ROLL3,MONTHS_OF_SUPPLY_ROLL6,MEDIAN_DOM_ROLL3,MEDIAN_DOM_ROLL6,PRICE_DROPS_ROLL3,PRICE_DROPS_ROLL6,PRICE_DROPS_YOY,RISK_SCORE,RISK_LABEL
0,2013-02-01,139000.0,5150.0,9615.0,29834.0,5.8,74.0,0.193739,2,2013,...,33930.666667,5.0,5.066667,70.666667,67.666667,0.201795,0.235197,-11.480097,0.914440,0
1,2013-03-01,155000.0,6877.0,10864.0,29598.0,4.3,67.0,0.202142,3,2013,...,32499.833333,5.3,5.233333,72.666667,69.166667,0.194595,0.220236,-23.461827,0.368770,0
2,2013-04-01,173000.0,7931.0,12783.0,30195.0,3.8,50.0,0.223249,4,2013,...,31176.333333,5.2,5.000000,72.000000,69.833333,0.208715,0.211570,-24.855574,0.036273,1


---
## 1. Define Features and Targets

### What are we doing in this cell?

In machine learning:
- **X** = the input features (everything the model is allowed to look at to make its prediction)
- **y** = the target (what the model needs to predict)

We **exclude** from X:
- `PERIOD_BEGIN` — it's a date used for splitting, not a numeric signal
- `RISK_SCORE` and `RISK_LABEL` — these are our targets, a model can't use the answer as its own input
- All current-month market columns — these values would not be available when predicting that month. We use their lag and rolling features instead, because those contain only past information.

**Data leakage** is when your model accidentally gets access to information it wouldn't have at prediction time. It causes the model to look great in training but fail in the real world.

In [30]:
CURRENT_MONTH_COLUMNS = ['MEDIAN_SALE_PRICE', 'HOMES_SOLD', 'NEW_LISTINGS',
                         'INVENTORY', 'MONTHS_OF_SUPPLY', 'MEDIAN_DOM',
                         'PRICE_DROPS']

EXCLUDE = ['PERIOD_BEGIN', 'RISK_SCORE', 'RISK_LABEL'] + CURRENT_MONTH_COLUMNS

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE]

X = df[FEATURE_COLS]
y_reg = df['RISK_SCORE']    # regression target
y_clf = df['RISK_LABEL']    # classification target

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'\nRegression target (RISK_SCORE):  {y_reg.describe().round(3).to_dict()}')
print(f'Classification target (RISK_LABEL): {y_clf.value_counts().to_dict()}')

Features (31): ['MONTH', 'YEAR', 'MEDIAN_SALE_PRICE_LAG_1', 'MEDIAN_SALE_PRICE_LAG_3', 'HOMES_SOLD_LAG_1', 'HOMES_SOLD_LAG_3', 'NEW_LISTINGS_LAG_1', 'NEW_LISTINGS_LAG_3', 'INVENTORY_LAG_1', 'INVENTORY_LAG_3', 'MONTHS_OF_SUPPLY_LAG_1', 'MONTHS_OF_SUPPLY_LAG_3', 'MEDIAN_DOM_LAG_1', 'MEDIAN_DOM_LAG_3', 'PRICE_DROPS_LAG_1', 'PRICE_DROPS_LAG_3', 'MEDIAN_SALE_PRICE_ROLL3', 'MEDIAN_SALE_PRICE_ROLL6', 'HOMES_SOLD_ROLL3', 'HOMES_SOLD_ROLL6', 'NEW_LISTINGS_ROLL3', 'NEW_LISTINGS_ROLL6', 'INVENTORY_ROLL3', 'INVENTORY_ROLL6', 'MONTHS_OF_SUPPLY_ROLL3', 'MONTHS_OF_SUPPLY_ROLL6', 'MEDIAN_DOM_ROLL3', 'MEDIAN_DOM_ROLL6', 'PRICE_DROPS_ROLL3', 'PRICE_DROPS_ROLL6', 'PRICE_DROPS_YOY']

Regression target (RISK_SCORE):  {'count': 155.0, 'mean': -0.112, 'std': 0.625, 'min': -1.569, '25%': -0.583, '50%': -0.054, '75%': 0.303, 'max': 1.18}
Classification target (RISK_LABEL): {0: 93, 1: 62}


---
## 2. Train / Test Split

### What are we doing in this cell?

We split the data into two sets:
- **Training set** — the model learns from this (80% of data)
- **Test set** — we evaluate the model on this (20% of data)

The test set acts like a final exam — the model has never seen it during training, so its performance on the test set tells us how well it will generalise to new, unseen data.

> **Important for time-series data:** We do a **chronological split** — the first 80% of months are training, the last 20% are test. We do NOT shuffle randomly. If we shuffled, the model could "see the future" (e.g. learn from 2024 data to predict 2020), which would make it look good but be useless in practice.

In [31]:
train_size= int(len(df) * 0.8)

X_train= X.head(train_size)
X_test = X.tail(len(X)- train_size)

y_reg_train= y_reg.head(train_size)
y_reg_test= y_reg.tail(len(y_reg)- train_size)

y_clf_train= y_clf.head(train_size)
y_clf_test= y_clf.tail(len(y_clf)- train_size)

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))




Training rows: 124
Test rows: 31


---
## 3. Feature Scaling

### What are we doing in this cell?

**Feature scaling** standardises all columns to have a mean of 0 and standard deviation of 1 (the same z-score normalisation we used for the risk score).

Why does this matter? Some features have very different scales:
- `INVENTORY` might be in the tens of thousands (e.g. 25,000)
- `MONTHS_OF_SUPPLY` is a small number (e.g. 4.5)

Without scaling, models like Logistic Regression and Linear Regression treat large-scale features as more important just because their numbers are bigger — even if they're not.

> **Critical rule:** We fit the scaler **only on the training set** and then apply it to both train and test. If we fit on all data, the scaler would "know" about test values, which is a form of data leakage.

In [32]:
scaler= StandardScaler()
X_train_sc= scaler.fit_transform(X_train)  #fit AND transform on train
X_test_sc= scaler.transform(X_test) # only transform on test (no fitting)

print('Scaling Complete')
print(f'Train mean(should be ~0): {X_train_sc.mean(axis=0).round(3)[:3]}...')
print(f'Train std(should be ~1): {X_train_sc.std(axis=0).round(3)[:3]}...')

Scaling Complete
Train mean(should be ~0): [-0.  0.  0.]...
Train std(should be ~1): [1. 1. 1.]...


---
## 4. Regression Models

### What are we doing in this cell (Linear Regression)?

**Linear Regression** is the simplest possible model. It assumes the relationship between inputs and output is a straight line:

`prediction = w₁×feature₁ + w₂×feature₂ + ... + b`

Where `w₁, w₂, ...` are **weights** (learned during training) and `b` is a **bias/intercept**.

Strengths: fast, interpretable, good baseline  
Weakness: can only capture linear (straight-line) relationships

Metrics we track:
- **R²** (R-squared) — how much of the variance in the target the model explains. Range 0–1; higher is better. R²=1 means perfect prediction.
- **RMSE** (Root Mean Squared Error) — average prediction error in the same units as the target. Lower is better.

In [33]:
lr= LinearRegression()
lr.fit(X_train_sc, y_reg_train)

y_pred_lr_train= lr.predict(X_train_sc)
r2_lr= r2_score(y_reg_train, y_pred_lr_train)
rmse_lr = np.sqrt(mean_squared_error(y_reg_train, y_pred_lr_train))
print(f'R2 : {r2_lr: .4f}')
print(f'RMSE: {rmse_lr : .4f}')



R2 :  0.9672
RMSE:  0.1154


### What are we doing in this cell (Random Forest Regressor)?

A **Random Forest** is an **ensemble** of many **Decision Trees**. Each decision tree learns a series of if/else rules (like a flowchart) to predict the target. The forest averages the predictions of all trees.

```
          Is MONTHS_OF_SUPPLY > 6?
         /                         \
       Yes                          No
  Is MEDIAN_DOM > 80?          Is PRICE_DROPS > 0.3?
     /        \                     /            \
  High Risk   Medium Risk      Medium Risk     Low Risk
```

Random Forests are powerful because:
- They handle non-linear relationships well
- They're resistant to overfitting (averaging many trees reduces noise)
- They naturally handle feature interactions

`n_estimators=100` means we grow 100 trees and average their predictions.

In [34]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_sc, y_reg_train)

y_pred_rfreg_train = rf_reg.predict(X_train_sc)
r2_rf = r2_score(y_reg_train, y_pred_rfreg_train)
rmse_rf = np.sqrt(mean_squared_error(y_reg_train, y_pred_rfreg_train))

print('── Random Forest Regressor (train set) ───────')
print(f'  R²:   {r2_rf:.4f}')
print(f'  RMSE: {rmse_rf:.4f}')

── Random Forest Regressor (train set) ───────
  R²:   0.9818
  RMSE: 0.0859


---
## 5. Classification Models

### What are we doing in this cell (Logistic Regression)?

Despite its name, **Logistic Regression** is a *classification* model (not regression). It predicts the **probability** that an observation belongs to class 1 (high risk), then converts that to a 0/1 label using a 0.5 threshold.

It's the classification equivalent of Linear Regression — simple, fast, and interpretable.

Classification metrics:
- **Accuracy** — % of predictions that were correct
- **Precision** — of all predicted high-risk months, what % were actually high risk?
- **Recall** — of all actual high-risk months, what % did we correctly identify?
- **F1 score** — harmonic mean of precision and recall (balanced metric)

In [35]:
log_clf= LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train_sc, y_clf_train)

y_pred_log_train= log_clf.predict(X_train_sc)
acc_log= accuracy_score(y_clf_train, y_pred_log_train)

print('── Logistic Regression (train set) ───────────')
print(f'  Accuracy: {acc_log:.4f}')
print(classification_report(y_clf_train, y_pred_log_train,
                             target_names=['Low Risk', 'High Risk']))



── Logistic Regression (train set) ───────────
  Accuracy: 0.9194
              precision    recall  f1-score   support

    Low Risk       0.91      0.94      0.92        62
   High Risk       0.93      0.90      0.92        62

    accuracy                           0.92       124
   macro avg       0.92      0.92      0.92       124
weighted avg       0.92      0.92      0.92       124



### What are we doing in this cell (Random Forest Classifier)?

Same idea as the Random Forest Regressor, but for classification. Instead of averaging numeric predictions, each tree in the forest **votes** for a class label, and the majority vote wins.

In [36]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_sc, y_clf_train)

y_pred_rfclf_train = rf_clf.predict(X_train_sc)
acc_rf = accuracy_score(y_clf_train, y_pred_rfclf_train)

print('── Random Forest Classifier (train set) ──────')
print(f'  Accuracy: {acc_rf:.4f}')
print(classification_report(y_clf_train, y_pred_rfclf_train,
                             target_names=['Low Risk', 'High Risk']))

── Random Forest Classifier (train set) ──────
  Accuracy: 1.0000
              precision    recall  f1-score   support

    Low Risk       1.00      1.00      1.00        62
   High Risk       1.00      1.00      1.00        62

    accuracy                           1.00       124
   macro avg       1.00      1.00      1.00       124
weighted avg       1.00      1.00      1.00       124



---
## 6. Save Models

### What are we doing in this cell?

We save all 4 trained models and the scaler to disk using **joblib**. This means:
- We don't have to re-train every time we open a new notebook
- The models can be loaded in notebooks 05 (evaluation) and 06 (predictions)

Think of it like saving a Word document — the trained model is stored as a `.pkl` (pickle) file.

In [37]:
os.makedirs('../models', exist_ok=True)

joblib.dump(lr,      '../models/linear_regression.pkl')
joblib.dump(rf_reg,  '../models/rf_regressor.pkl')
joblib.dump(log_clf, '../models/logistic_regression.pkl')
joblib.dump(rf_clf,  '../models/rf_classifier.pkl')
joblib.dump(scaler,  '../models/scaler.pkl')
joblib.dump(FEATURE_COLS, '../models/feature_cols.pkl')

print('Saved models:')
for f in os.listdir('../models'):
    if f.endswith('.pkl'):
        size_kb = os.path.getsize(f'../models/{f}') / 1024
        print(f'  {f:<35} {size_kb:.1f} KB')

Saved models:
  scaler.pkl                          2.3 KB
  linear_regression.pkl               1.0 KB
  rf_regressor.pkl                    1128.8 KB
  logistic_regression.pkl             1.1 KB
  feature_cols.pkl                    0.6 KB
  rf_classifier.pkl                   231.4 KB


---
## Summary

| Model | Task | Training Metric |
|-------|------|-----------------|
| Linear Regression | Regression (predict RISK_SCORE) | R² and RMSE printed above |
| Random Forest Regressor | Regression (predict RISK_SCORE) | R² and RMSE printed above |
| Logistic Regression | Classification (predict RISK_LABEL) | Accuracy + F1 printed above |
| Random Forest Classifier | Classification (predict RISK_LABEL) | Accuracy + F1 printed above |

> **Note:** High training accuracy doesn't guarantee the model works well on new data. That's what notebook 05 is for.

**Next step →** Open `05_model_evaluation.ipynb` to test all models on held-out data.